# Введение

# План семинара
Сегодняшний план такой:

* Постановка задачи
* Обработка данных
    * Знакомство с данными
    * Поиск data leakage
    * Обработка пропущенных значений
    * Обработка категориальных значений
    * Поиск выбросов
* Применение KNN на полученных данных

# Бизнес-задача

## Цена квартиры
<img src="https://i.ibb.co/yfxgLRF/photo-2022-07-16-19-34-07.jpg" alt="photo-2022-07-16-19-34-07" border="0" height=700>
<img src="https://i.ibb.co/GQWhmNw/photo-2022-07-16-19-34-26.jpg" alt="photo-2022-07-16-19-34-26" border="0" height=700>

Уметь оценивать стоимость недвижимости может быть ужасно выгодно. Первые применения, которые приодят в голову:
* Помощь покупателям в выборе квартиры, как это делает ЦИАН
* Помощь продавцам в установке справедливой цены, как это делает ЦИАН
* Поиск недооцененных квартир, для покупки и перепродажи

Мы видим, что хорошо сделанная модель дл яоценки стоимости квартиры может принести огромную прибыль ее создателю. Давайте попробуем применить полученные нами знания о пайплайне МО для решения этой задачи.

# Постановка задачи

Данные мы опять же возьмем с сайта https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot?resource=download
Данные о продажах квартир были получены с помощью скрейпинга сайта Domain.com.au (скрейпинг - загрузка веб сайтов внутри кода и извлечение оттуда информации). Все квартиры продаются в Мельбурне.
У нас есть много разнородных признаков и одна целевая переменная - Price.

# Обработка данных

In [ ]:
!unzip archive.zip

In [ ]:
import pandas as pd

data = pd.read_csv('melb_data.csv')
data

Посмотрим на все доступные колонки


In [ ]:
X = data.drop(columns='Price')
y = data['Price']

In [ ]:
X

Всего у нас 20 признаков и 13580 обучающих объектов


## Train/validation/test split не нужен

Сейчас мы вообще не будем делить наши данные.

Тестовую часть, как и в предыдущем ноутбуке, мы опустим. Хотя важно помнить, что она критически важная для моделей, которые вы будете строить для реальных прменений!

Валидационная часть в данном случае не нужна, так как промежуточные метрики во время разработки модели мы будем отслеживать по кросс-валидации, которая не требует отдельных валидационнных данных.

<img src="https://i.ibb.co/xC3vXT8/2022-07-16-20-06-36.png" alt="2022-07-16-20-06-36" border="0">

## Найдем лики

В следующем шаге мы продйемся по всем признакам и посмотрим на них подрбонее, когда будем делить на численные категориальные итд. Сейчас нас интересует только один признак -- Method. Вот такое описание у него на страничке, с которой мы загрузили наши данные.

Method: S - property sold; SP - property sold prior; PI - property passed in; PN - sold§ prior not disclosed; SN - sold not disclosed; NB - no bid; VB - vendor bid; W - withdrawn prior to auction; SA - sold after auction; SS - sold after auction price not disclosed. N/A - price or highest bid not available.

Если в кратце, то S значит, что продажа случилась, SP что продажа случилась раньше, чем указанная дата, SN что не известно, случилась ли продажа итд. Понятно, что этот признак не будет доступен во время применения нашей модели.

In [ ]:
X = X.drop(columns='Method')

# Найдем пропущеные значения

In [ ]:
import numpy as np

np.nan # - так обычно выглдят пропущенные значения

In [ ]:
X.isna()

Пропущенные значения в табличках могут быть не только np.nan, но любыми другими значениями, которые сильно сложнее фильтровать. Например, вместо пропущенного значения может стоять строчка "missing". Сейчас мы не будем рассматривать такую возможность и понадеемся на того, кто собирал данные.

In [ ]:
X.isna().sum()

Теперь нам нужно решить, что делать с колонками. Прочитаем их описание

* Car: Number of carspots

Число пропущенных парковочных мест очень мало. Попробуем заменить все пропущенные значения на среднее по датасету.

In [ ]:
#хотим выполнить X.loc[X['Car'].isna(), 'Car'] = X['Car'].mean()

In [ ]:
X['Car'].isna()

In [ ]:
X.loc[X['Car'].isna(), 'Car']

In [ ]:
X.loc[X['Car'].isna(), 'Car'] = X['Car'].mean()


* BuildingArea: Building Size

Это значение отсутствует у половины примеров в нашем датасете. Скорее всего это очень важный признак, но у нас нет никакой надежды достать его реальное значение для половины обхектов.


In [ ]:
X = X.drop(columns='BuildingArea')


* CouncilArea: Governing council for the area

Пропущено достаточно много значений признака. Понятно, что мы можем посмотреть на ближайшие дома, определив их по долготе и широте, а потом взять их значение CouncilArea, но признак вряд ли очень полезный, поэтому мы его выкинем.


In [ ]:
X = X.drop(columns='CouncilArea')


* YearBuilt: not specified

Дата постройки отсутсвтует у достаточно большого числа объявлений. Мы заменим значение на среднее по датасету + добавим новую бинарную колонку, в которую пропишем 1, если YearBuilt не был указан. Таким образом, мы избавимся от пропущенного значения, не выкинем колонку или много данных и при этом сохраним информацию о том, что у некоторых обхектов отсутствовал признак.

In [ ]:
X['YearBuiltWasMissing'] = X['YearBuilt'].isna()
X.loc[X['YearBuilt'].isna(), 'YearBuilt'] = X['YearBuilt'].mean()

# Определение типов признаков

In [ ]:
list(zip(X.columns, X.dtypes))

dtype('O') означает, что в колонке находятся какие-то Python объекты. В больинстве случаев это значит, что там лежат строки.

dtype('float64') означает, что в колонке находятся числа с плавающей точкой

dtype('bool') означает, что в колонке находятся True или False

In [ ]:
X

In [ ]:
categorical_cols = [
    'Suburb',
    'Type',
    'Regionname',
    'YearBuiltWasMissing'
]

numerical_cols = [
    'Rooms',
    'Distance',
    'Bedroom2',
    'Bathroom',
    'Car',
    'Landsize',
    'YearBuilt',
    'Lattitude',
    'Longtitude',
    'Propertycount'
]


cols_to_drop = [
    'Address',
    'SellerG',
    'Date',
    'Postcode'
]

У нас есть сомнения по поводу того, включать ли Suburb как категориальный признак. Давайте посмотрим, сколько возможных значений существует.

In [ ]:
# Исследование колонок Address  Postcode  Suburb

In [ ]:
suburb_counts = X['Suburb'].value_counts()
suburb_counts

In [ ]:
is_rare_suburb = X['Suburb'].apply(lambda x: suburb_counts[x] < 200)
X.loc[is_rare_suburb, 'Suburb'] = 'RareSuburb'

In [ ]:
X = X.drop(columns=cols_to_drop)

## One-hot-encoding

In [ ]:
X = pd.get_dummies(X, columns=categorical_cols)

In [ ]:
X

# Поиск выбросов

In [ ]:
import matplotlib.pyplot as plt

for ax, num_col_name in zip(plt.subplots(4, 3, figsize=(20,20))[1].flatten(), numerical_cols):
    ax.set_title(num_col_name)
    ax.boxplot(X[num_col_name])

Если бы мы обучали алгоритм вроде LinearRegression, про который вы узнаете позже, то нам пришлось бы убирать выбросы, но мы будем использовать KNN, который достаточно хорошо себя ведет и при наличии выбросов.

## Standard Scaler

На последок, мы проведем стандартизацию признаков, про которую говорили раньше. Сейчас мы сдлеаем очень плохую и неправильную ошибку, чтобы не загромождать сегодняшнее видео. Мы посчитаем среднее и дисперсси признаков по всему датасету, а не только по обучающей части.

Как вы помните, при кроссвалидации мы выделяем обучающую и валидационную часть только в самом конце, поэтмоу проведя стандартизацию сейчас мы создадим утечку данных. Чтобы этой утечки не допускать нужно использовать sklearn.piplines или пользоваться первым метдом проведение кросс валидации, про который мы поговорим чреез KFold.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Обучение KNN и кроссвалидация

Напомним, как работает кроссвалидация
<img src="https://i.ibb.co/xC3vXT8/2022-07-16-20-06-36.png" alt="2022-07-16-20-06-36" border="0">


Для првоедения кросс валидации есть несколько путей.

Первый, который мы попробуем, это прямое использование sklearn.model_selection.KFold, который выдаст нам разделения на трейн и тест.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor(5)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

cv = KFold(n_splits=5)
errors = []

for train_idx, val_idx in tqdm(cv.split(X)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    knn.fit(X_train, y_train)
    pred_val = knn.predict(X_val)
    errors.append(
        mean_squared_error(y_val, pred_val)
    )

print()
print('Metrics')
print(errors)
print('RMSE=', np.mean(errors) ** 0.5)

Есть также второй более простой способ -- использование cross_val_score

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(knn, X, y, cv=5)
scores

cross_val_score выдает стандартный скор для всех регрессоров -- R_squared. Но мы не хотим его, поэтому передадим свой собтсвенный скорер

In [ ]:
from sklearn.metrics import make_scorer
scorer = make_scorer(lambda y_true, y_pred: mean_squared_error(y_true, y_pred),
                     greater_is_better=False)
errors = cross_val_score(knn, X, y, cv=5, scoring=scorer)

print()
print('Metrics')
print(errors)
print('RMSE=', np.mean(-errors) ** 0.5)

# Финальное обучение

In [ ]:
knn.fit(X, y)